# Species Occurrence Records and Distribution Models for Sub-Saharan African Bats Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, overview, extraction, and exploration of the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset is structured using the Croissant schema and accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.fa9y-hynm/fair2.json`

Please follow the steps below to explore the metadata, record sets, fields, and columns, referencing all entities by their `@id` values as per best practice.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.fa9y-hynm/fair2.json'

# Load Croissant metadata and dataset
dataset = mlc.Dataset(croissant_url)
# Metadata as JSON-LD
metadata = dataset.metadata.to_json()

print("Dataset Name:", metadata.get('name', 'N/A'))
print("Description:", metadata.get('description', 'N/A'))

# Optional: Print primary keys for record sets if available
# Record sets (entities) are referenced by their `@id` field.

## 2. Data Overview
Review available record sets, fields, and their IDs.

Record sets and their fields are central to organizing the dataset. For each record set you should reference its `@id`, for fields and columns use their respective `@id`s.

In [ ]:
# List all record sets, their IDs, and summary fields
record_sets = []
if hasattr(dataset, 'record_sets'):
    for rs in dataset.record_sets:
        print(f"RecordSet: {rs['@id']} (name: {rs['name']})")
        record_sets.append(rs['@id'])
        for field in rs.get('fields', []):
            print(f"    Field: {field['@id']} (name: {field.get('name', '')})")
        for column in rs.get('columns', []):
            print(f"    Column: {column['@id']} (name: {column.get('name', '')})")
else:
    # If dataset.recordSets is not a property, attempt to load from metadata
    for rs in metadata.get('recordSet', []):
        print(f"RecordSet: {rs['@id']}")
        record_sets.append(rs['@id'])
        for field in rs.get('field', []):
            print(f"    Field: {field['@id']} (name: {field.get('name', '')})")
        for column in rs.get('column', []):
            print(f"    Column: {column['@id']} (name: {column.get('name', '')})")

# Show a sample of records (using the first record set by its `@id`)
if record_sets:
    print("\nSample records from first RecordSet:")
    for x in dataset.records(record_set=record_sets[0]):
        print(x)
        break  # just show one record for preview

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing each by its `@id`.
You can use these DataFrames for further processing and visualization.

In [ ]:
# Extract data from all available record sets
# Each record set is referenced via its @id
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded RecordSet {rs_id} with columns: {df.columns.tolist()}")

# Preview the first RecordSet DataFrame
if record_sets:
    main_rs_id = record_sets[0]
    print(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by criteria, normalizing numeric fields, or grouping by categories.

Make sure you reference fields and columns uniquely by their `@id` as shown below.

In [ ]:
# Example EDA: Filter, normalize and group

# Select RecordSet and fields to analyze
# Replace these with actual field/column @ids you found above
example_rs_id = main_rs_id  # Use the first RecordSet for demonstration
df = dataframes[example_rs_id]

# Print available columns for guidance
print("Available fields for analysis:")
print(df.columns.tolist())

# Replace with actual numeric field column @id from above (e.g. '@id': 'cr:extentOfOccurrence')
numeric_field_id = None
for col in df.columns:
    # Try to pick a numeric column automatically (otherwise user can insert manually)
    if df[col].dtype in ['float64', 'int64']:
        numeric_field_id = col
        break
if not numeric_field_id:
    numeric_field_id = df.columns[0]  # fallback

# Set threshold for filtering
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping (pick a categorical column by @id)
group_field_id = None
for col in df.columns:
    if df[col].dtype == 'object' and df[col].nunique() < 50:
        group_field_id = col
        break
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships in the dataset.

Below, we show an example histogram and scatterplot using matplotlib. Make sure to reference the fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of numeric field
plt.figure(figsize=(8,5))
filtered_df[numeric_field_id].hist(bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Scatter plot (if at least two numeric columns)
numeric_columns = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
if len(numeric_columns) >= 2:
    plt.figure(figsize=(8,5))
    plt.scatter(filtered_df[numeric_columns[0]], filtered_df[numeric_columns[1]], alpha=0.6)
    plt.xlabel(numeric_columns[0])
    plt.ylabel(numeric_columns[1])
    plt.title(f"Scatter plot: {numeric_columns[0]} vs {numeric_columns[1]}")
    plt.show()

## 6. Conclusion
This notebook demonstrated loading the FAIR^2 bat species dataset using `mlcroissant`, with clear referencing of entities by their `@id`. You reviewed available record sets, fields, and columns, loaded them into DataFrames, applied basic EDA, and visualized key variables.

Further analysis could focus on deeper ecological, biogeographical, or conservation metrics, leveraging the rich metadata and structure provided by the Croissant schema.